# RAG: Semantic Clustering

This notebook clusters an existing vector database collection by embedding similarity,
labels each cluster with an LLM, and indexes cluster summaries as a navigational layer.
Run example_RAG_01_load.ipynb first to populate the 'books' collection.

## Initialize

In [ ]:
from agentic_patterns.core.vectordb import get_vector_db, vdb_add
from agentic_patterns.core.vectordb.clustering import cluster
from agentic_patterns.agents.rag.clustering import label_clusters

In [ ]:
vdb = get_vector_db("books")
count = vdb.count()
assert count > 0, "Vector database is empty. Run example_RAG_01_load.ipynb first."
print(f"Collection has {count} documents")

## Cluster with HDBSCAN

HDBSCAN discovers the number of clusters automatically.
Points in sparse regions get label -1 (noise) and are not forced into a cluster.

In [ ]:
result = cluster(vdb, algorithm="hdbscan")

print(f"Found {len(result.clusters)} clusters")
for c in result.clusters:
    label = "(noise)" if c.cluster_id == -1 else f"cluster {c.cluster_id}"
    print(f"  {label}: {len(c.items)} items")

## Label clusters with LLM

Each cluster gets a short label and a summary sentence based on a sample of its items.

In [ ]:
labeled = await label_clusters(result)

for c in labeled.clusters:
    if c.cluster_id == -1:
        print(f"  Noise cluster: {len(c.items)} unclustered items")
        continue
    print(f"\nCluster {c.cluster_id}: {c.label}")
    print(f"  Summary: {c.summary}")
    print(f"  Size: {len(c.items)} items")
    print(f"  Sample: {c.items[0].text[:120]}...")

## Cluster with K-Means (fixed number of clusters)

When the desired number of topics is known, k-means produces balanced clusters.

In [ ]:
result_km = cluster(vdb, algorithm="kmeans", n_clusters=8)
labeled_km = await label_clusters(result_km)

print("K-Means clusters (k=8):")
for c in labeled_km.clusters:
    print(f"  [{c.cluster_id}] {c.label} ({len(c.items)} items)")

## Map clusters to predefined criteria

Comparing cluster labels against a set of criteria reveals corpus coverage.

In [ ]:
criteria = [
    "Ethical decision-making under pressure",
    "Resource management and survival",
    "Communication with unknown entities",
    "Character relationships and conflict",
    "World-building and setting",
]

print("Cluster labels from corpus:")
for c in labeled.clusters:
    if c.cluster_id != -1:
        print(f"  - {c.label}")

print("\nPredefined criteria:")
for cr in criteria:
    print(f"  - {cr}")

print("\n(A similarity comparison between cluster labels and criteria would identify gaps)")

## Index cluster summaries as a navigational layer

Storing cluster labels and summaries in a separate collection enables
two-stage retrieval: find the relevant cluster, then retrieve within it.

In [ ]:
vdb_index = get_vector_db("books_cluster_index")

added = 0
for c in labeled.clusters:
    if c.cluster_id == -1:
        continue
    doc_ids = ",".join(item.doc_id for item in c.items)
    text = f"{c.label}: {c.summary}"
    meta = {"cluster_id": c.cluster_id, "item_count": len(c.items), "doc_ids": doc_ids}
    result = vdb_add(vdb_index, text=text, doc_id=f"cluster-{c.cluster_id}", meta=meta, force=True)
    if result:
        added += 1

print(f"Indexed {added} cluster summaries into 'books_cluster_index'")

## Two-stage retrieval via cluster index

First find the relevant cluster, then retrieve chunks from within that cluster only.

In [ ]:
from agentic_patterns.core.vectordb.retrieval import retrieve

query = "Characters facing moral dilemmas"

# Stage 1: find the best matching cluster
cluster_hits = retrieve(vdb_index, query=query, max_results=2)
print("Matching clusters:")
for hit in cluster_hits:
    print(f"  score={hit.score:.3f} | {hit.text}")

# Stage 2: retrieve within the best cluster's doc_ids
if cluster_hits:
    best_cluster_meta = cluster_hits[0].metadata
    doc_id_list = best_cluster_meta.get("doc_ids", "").split(",")
    print(f"\nRetrieving from {len(doc_id_list)} chunks in this cluster")
    
    # Direct fetch of cluster items from the main collection
    cluster_docs = vdb.get(ids=doc_id_list[:20], include=["documents", "metadatas"])
    for doc_id, doc in zip(cluster_docs["ids"][:3], cluster_docs["documents"][:3]):
        print(f"  [{doc_id}] {doc[:100]}...")